# **Algiers in 24 Hours - Tourist Optimizer**

---
# **SECTION 0 - Imports and Global Configuration**

In [ ]:
TMAX_FULL = 720
TMAX_HALF = 360
STARTING_TIME = 480



## Imports

In [ ]:
import random
import math
from typing import List, Dict, Any

---
# **SECTION 1 - Shared Data Structures**
### All 6 Members

## 1.1 - Location Class

In [ ]:
class Landmark:
    """
    Represents a single tourist landmark or attraction.
    
    Attributes:
        id (int): A unique numerical identifier for the landmark.
        name (str): The name of the landmark.
        lon (float): Longitude coordinate.
        lat (float): Latitude coordinate.
        interest_score (float): The rating or user interest score.
        opening_hours (dict[str, list[int]]): A dictionary mapping days of the week 
            (e.g., 'Monday') to a list of 24 integers representing hours. 
            1 means open, 0 means closed. Example: {'Monday': [0,0,...,1,1,0]}
        visit_duration (int): Estimated time spent at the landmark in minutes.
        landmark_type (str): The category of the landmark (e.g., 'Museum', 'Park').
    """

    def __init__(self, id: int, name: str, lon: float, lat: float, 
                 interest_score: float, opening_hours: dict[str, list[int]], 
                 visit_duration: int, landmark_type: str):
        
        self.id = id
        self.name = name
        self.lon = lon
        self.lat = lat
        self.interest_score = interest_score
        self.opening_hours = opening_hours
        self.visit_duration = visit_duration
        self.landmark_type = landmark_type

    def is_open(self, day: str, arrival_minutes: float) -> bool:
        """
        Checks if the landmark is open for the entire visit duration.

        Args:
            day (str): Day abbreviation e.g. 'mon', 'tue'
            arrival_minutes (float): Arrival time in minutes from midnight
                                    e.g. 540 = 09:00, 545.7 = 09:05:42

        Returns:
            bool: True if landmark is open for every hour slot of the visit
        """
        opening = self.opening_hours[day]

        if opening is None:
            return False

        finish_minutes = arrival_minutes + self.visit_duration

        # which hour slot do we arrive in?
        arrival_hour = int(arrival_minutes // 60)

        # which hour slot are we still INSIDE at the end?
        # subtract 1 so that finishing exactly on the hour (e.g. 600 = 10:00)
        # does not count as being inside the next slot (hour 10)
        finish_hour = int((finish_minutes - 1) // 60)

        for hour in range(arrival_hour, finish_hour + 1):
            if opening[hour % 24] == 0:
                return False

        return True


In [ ]:
#this class represent single hotel

class Hotel:

    def __init__(self , name , lon, lat ):
      self.id = name
      self.lon = lon
      self.lat = lat 


## 1.2 - Problem Class

## Class for informed search algorithms

In [ ]:
# This class holds everything the algorithms need and give clean access to it.

## Class for local search algorithms


In [ ]:
class TravelProblem_LocalSearch:
    """
    This class represents the travel guide problem formulation for Local Search algorithms.
    It manages state validation, neighbor generation, and state evaluation for a 24-hour trip.
    """

    def __init__(self, landmarks: List['Landmark'], travel_information: Dict[str, Any]):
        """
        Initializes the travel problem with landmarks and user preferences.

        Args:
            landmarks: A list of Landmark objects available to visit.
            travel_information: A dictionary containing user preferences and constraints.
                Expected keys:
                - 'hotel': Hotel object (starting and ending point).
                - 'Travel_day': three lettter string  (e.g., 'Mon' , 'Fri' ).
                - 'Travel_Time': float (Max travel time allowed in hours).
                - 'Landmarks_number': int (Number of landmarks the user wants to visit).
                - 'type_filter': list[str] (Allowed categories, e.g., ['Museum', 'Park']).
                - 'time_matrix': dict (Precomputed travel times in minutes between locations).
        """
        if not landmarks:
            raise ValueError("No list of landmarks provided!")
        
        self.landmarks = landmarks
        self.landmarks_list = [landmark.name for landmark in landmarks]
        
        # Extract user preferences and constraints
        self.hotel = travel_information['hotel']
        self.Travel_day = travel_information['Travel_day']
        self.max_travel_time = travel_information['Travel_Time']  
        self.Landmarks_number = travel_information['Landmarks_number']
        self.type_filter = travel_information['type_filter']
        
        # Important: The time matrix must be passed in to calculate real road travel times
        self.time_matrix = travel_information['time_matrix']

        # Generate the initial starting state (must be done AFTER setting rules)
        self.initial_state = self._generate_random_state()


    def valid_state(self, state: List['Landmark'], hard_constraints: bool = True) -> bool:
        """
        Validates an itinerary based on time limits, opening hours, and category filters.
        """
        # 1. Quick Failure Checks: Null values or duplicate landmarks
        if not state:
            return False
        if None in state: 
            return False
        if len(set(lm.id for lm in state)) != len(state): 
            return False
        
        trip_start_time = 8.0  # Assuming trips start at 8:00 AM and we can change it if we want

        #current time with minutes  
        current_time  = trip_start_time*60
        
        # 2. Iterate through the itinerary to track time and check constraints
        for i, landmark in enumerate(state):
            
            # Add travel time to get to this landmark
            if i == 0:
                travel_mins = self.time_matrix[self.hotel.id][landmark.name]
            else:
                travel_mins = self.time_matrix[state[i-1].name][landmark.name]
                           
             #arrival time in minutes 
            current_time += travel_mins

            # Check if landmark is open at arrival and exit times  
            if not landmark.is_open(self.Travel_day, current_time ): 
                return False
            
            if self.type_filter and landmark.landmark_type not in self.type_filter: 
                return False
                
            # Add the duration spent visiting the landmark ( in minutes )
            current_time += landmark.visit_duration 

        # 3. Add the return trip to the hotel
        return_mins = self.time_matrix[state[-1].name][self.hotel.id]
        
        current_time += return_mins 
        
        return_hour = current_time/60
        # 4. Hard Constraint: Did the total trip exceed the user's allowed time?
        if hard_constraints:
            if (return_hour - trip_start_time) > self.max_travel_time:
                return False
            
        return True
    


    def _generate_random_state(self) -> List['Landmark']:
        """
        Generates a valid, random initial state to kick off the search.
        
        """
        # using the landmark number if provided 
        if self.Landmarks_number : 
            state = [None for _ in range(self.Landmarks_number)]

            attempts = 0

            # Keep rolling until we find a combination that passes all constraints
            while not self.valid_state(state) and attempts < 1000:
                state = [random.choice(self.landmarks) for _ in range(self.Landmarks_number)] 
                attempts+=1

            if (attempts >  1000):raise ValueError("constraints for generation invalid")


        # without using the landmark number    
        else :
            state = []

            failure = 0 
            while failure < 10: 
                try_state =  state[:]
                item = random.choice(self.landmarks)
                if item not in state : try_state.append(item)
                if not self.valid_state(try_state) : 
                    failure +=1 
                    continue
                
                state  = try_state                      


        return state
    

    
    def generate_neighbors(self, state: List['Landmark']) -> List[List['Landmark']]:
        """
        Generates all legally valid neighbors using two strategies:
        1. Replacement: Swapping an existing landmark for an unused one.
        2. Internal Swap: Changing the order of current landmarks.
        3. If the itinerary size is dynamic, also uses Add and Remove.
        """
        neighbors = set()
        current_ids = [landmark.id for landmark in state]

        # --- Strategy 1: Replacement Neighbors ---
        for i, landmark in enumerate(state):
            for new_item in self.landmarks:
                if new_item.id not in current_ids:
                    state_copy = state[:]  
                    state_copy[i] = new_item
                    
                    if self.valid_state(state_copy): 
                        neighbors.add(tuple(state_copy))

                        

        # --- Strategy 2: Internal Swap Neighbors ---
        for i in range(len(state)):
            for j in range(i + 1, len(state)):
                state_copy = state[:]
                # Swap the positions
                state_copy[i], state_copy[j] = state_copy[j], state_copy[i]
                
                if self.valid_state(state_copy):
                    neighbors.add(tuple(state_copy))


        # --- Dynamic Strategies: ADD and REMOVE ---
        if not self.Landmarks_number:
            
            # --- Strategy 3: ADD a landmark ---
            for new_item in self.landmarks:
                if new_item.id not in current_ids:
                    # Try inserting the new item at every possible step of the trip
                    # range(len(state) + 1) allows us to put it at the very start, middle, or very end
                    for insert_index in range(len(state) + 1):
                        state_copy = state[:]
                        state_copy.insert(insert_index, new_item)
                        
                        if self.valid_state(state_copy):
                            neighbors.add(tuple(state_copy))

            # --- Strategy 4: REMOVE a landmark ---
            # We should only allow removing if the trip has more than 1 stop left!
            if len(state) > 1:
                for i in range(len(state)):
                    state_copy = state[:]
                    state_copy.pop(i)  # Remove the landmark at index i
                    
                    if self.valid_state(state_copy):
                        neighbors.add(tuple(state_copy))            

        return [list(s) for s in neighbors]
    


    def _generate_random_neighbor(self, state: List['Landmark']) -> List['Landmark']:
        """Returns one random valid neighbor from the current state."""
        neighbors = self.generate_neighbors(state)
        # Fallback if no neighbors are found (rare, but prevents crashes)
        if not neighbors:
            return state 
        return random.choice(neighbors)
    


    def evaluate(self, state: List['Landmark']) -> float:
        """ 
        The Fitness Function. 
        Minimizes distance while heavily maximizing the interest score.
        A lower resulting float indicates a better itinerary.
        """
        total_rating = 0
        total_travel_time = 0 

        for i, landmark in enumerate(state):
            total_rating += landmark.interest_score
            
            # Travel time from hotel to first landmark, or between landmarks
            if i == 0:
                total_travel_time += self.time_matrix[self.hotel.id][landmark.name]
            else:
                
                total_travel_time += self.time_matrix[state[i-1].name][landmark.name]

        # Travel time  from final landmark back to the hotel
        total_travel_time += self.time_matrix[state[-1].name][self.hotel.id]

       
        #let the rating as the most importatn criteria 
        #the lower is better , it will be negative
        score =  total_travel_time -1000*total_rating
        
        return score
    
    def distance(self, loc1: Any, loc2: Any) -> float:
        """
        Calculates the Great-Circle (Haversine) distance
        Accepts both Landmark and Hotel objects (duck typing) since both have .lat and .lon.
        """
        
        # radius of earth in kilometers 
        R = 6371.0
        lat1 = math.radians(loc1.lat)
        lon1 = math.radians(loc1.lon)
        lat2 = math.radians(loc2.lat)
        lon2 = math.radians(loc2.lon)

        dlon = lon2 - lon1
        dlat = lat2 - lat1

        a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos( lat2) * math.sin(dlon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        distance_km = R * c
        
        return distance_km


## 1.3 - Solution Class

In [ ]:
# This class store the result of an algorithm and validate it.

---
# **SECTION 2 - Data Collection**

## 2.1 - Landmarks Dataset


We gathered - clean - data about  **+60 attractions/landmarks in Algiers**, including the following attributes:  
* __Attraction/landmark name.__
* __Description (__ short description __)__  
* __Type of attraction/landmark (__ Monument, Park/Garden, Historical site ... etc __)__  
* __Rating (/10)__ 
* __GPS coordinates (__ longitude, latitude __)__
* __Estimated visit duration (__ in minutes  __)__
* __Opening Hours__
* __Images__


In [1]:
import csv
from collections import Counter

# path to the landmarks CSV file
LANDMARK_PATH = "./../dataset/landmarks/Algiers_landmarks.csv"

with open(LANDMARK_PATH, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

ratings = [float(r['Rating']) for r in rows]
types = Counter(r['Type'] for r in rows)


print(f"Total landmarks : {len(rows)}")
print(f"Average rating  : {sum(ratings)/len(ratings):.2f}")
print(f"Highest rating  : {max(ratings)}")
print(f"Lowest rating   : {min(ratings)}")
print(f"Unique types    : {len(types)}")


print("\nCount by type: ")
for t, c in types.most_common():
    print(f"  {t:<25} {c}")


print("\nTop 10 landmarks: ")
for i, r in enumerate(sorted(rows, key=lambda x: float(x['Rating']), reverse=True)[:10], 1):
    print(f"  {i:>2}. {r['Rating']}  {r['Name']}")

print("\nBottom 5 landmarks: ")
for i, r in enumerate(sorted(rows, key=lambda x: float(x['Rating']))[:5], 1):
    print(f"  {i}. {r['Rating']}  {r['Name']}")

Total landmarks : 59
Average rating  : 7.50
Highest rating  : 9.7
Lowest rating   : 3.8
Unique types    : 11

Count by type: 
  Cultural Center & Event Venue 11
  Museum                    9
  Mosque                    7
  Park                      6
  Historical Site           5
  Public Square             5
  Nature                    4
  Beach                     4
  Shopping/Mall             4
  Monument                  2
  Cathedral                 2

Top 10 landmarks: 
   1. 9.7  Great Mosque of Algiers (Djamaa el Djazaïr)
   2. 9.6  Botanical Garden Hamma
   3. 9.5  Martyrs' Memorial (Maqam E'chahid)
   4. 9.5  Casbah of Algiers
   5. 9.3  La Grande Poste
   6. 9.2  Palais des Raïs (Bastion 23)
   7. 9.2  Garden City Mall
   8. 9.1  Notre Dame d'Afrique
   9. 9.1  Ketchaoua Mosque
  10. 9  National Museum of Fine Arts

Bottom 5 landmarks: 
  1. 3.8  Monument to the Dead of Algiers
  2. 4  Zeralda Beach
  3. 4  Dounia Park
  4. 4.2  Ben Aknoun Zoo
  5. 4.5  El Kettani Beach


## 2.2 - Hotels Dataset


We gathered a dataset of __+180 hotels__ across the Algiers region, collected from Google Maps.

In [ ]:
HOTELS_PATH = "./../dataset/hotels/Algiers_hotels.csv"

with open(HOTELS_PATH, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
 
print(f"Total hotels: {len(rows)}")

Total hotels: 184


## 2.2 - Time Matrix


Generated via the OSRM public API (router.project-osrm.org) using the driving profile.

- `time_matrix.npy` — (247 × 247) numpy array of travel times in minutes
- `time_matrix_named.csv` — same matrix with landmark/hotel names as row and column headers

Rows and columns are ordered: landmarks first (83), hotels second (184).

**Note:** The public OSRM API enforces a 100-coordinate limit per request,
so the matrix is built in batches. Results reflect road-network driving time
and may not account for real-time traffic.

In [ ]:
import pandas as pd
import numpy as np
import requests

landmarks = pd.read_csv("./../dataset/landmarks/final_dataset.csv")
hotels    = pd.read_csv("./../dataset/hotels/Algiers_hotels.csv")

names = list(landmarks["Name"]) + list(hotels["name"])

all_coords = (
    [(row["Longitude"], row["Latitude"]) for _, row in landmarks.iterrows()] +
    [(row["longitude"], row["latitude"]) for _, row in hotels.iterrows()]
)

OSRM_BASE = "http://router.project-osrm.org/table/v1/driving"
BATCH     = 100  # public OSRM hard limit

n = len(all_coords)
time_matrix = np.zeros((n, n), dtype=float)

for i in range(0, n, BATCH):
    for j in range(0, n, BATCH):
        src_slice = all_coords[i:i+BATCH]
        dst_slice = all_coords[j:j+BATCH]

        combined = src_slice + dst_slice
        coords_str = ";".join(f"{lon},{lat}" for lon, lat in combined)

        sources = ";".join(str(k) for k in range(len(src_slice)))
        destinations = ";".join(str(k + len(src_slice)) for k in range(len(dst_slice)))

        r = requests.get(
            f"{OSRM_BASE}/{coords_str}",
            params={"annotations": "duration", "sources": sources, "destinations": destinations},
            timeout=120
        )
        r.raise_for_status()
        block = np.array(r.json()["durations"], dtype=float) / 60  # seconds → minutes

        time_matrix[i:i+len(src_slice), j:j+len(dst_slice)] = block

np.save("time_matrix.npy", time_matrix)

df = pd.DataFrame(time_matrix.round(1), index=names, columns=names)
df.to_csv("time_matrix_named.csv")

print(f"Done. Matrix shape: {time_matrix.shape}")

The CSV file `time_matrix_named.csv` is converted into a JSON file `time_matrix.json`

In [ ]:
import pandas as pd
import json

# load the existing matrix CSV
df = pd.read_csv("time_matrix_named.csv", index_col=0)

# convert to nested dict {start: {end: minutes}}
time_dict = {}
for start in df.index:
    time_dict[start] = {}
    for end in df.columns:
        time_dict[start][end] = round(float(df.loc[start, end]), 1)

# save as JSON
with open("time_matrix.json", "w", encoding="utf-8") as f:
    json.dump(time_dict, f, ensure_ascii=False, indent=2)

print("Done. Saved to time_matrix.json")

---
# **SECTION 3 - Algorithms Representation**


## Greedy , Simulated Anealing , GA + optional Ones


code impelmentation + definition of each algorithm and of each parameter


---
# **SECTION 4 - Testing Data on ALgorithms**

---
# **SECTION 6 - Final Results and Submission**
### Person 1 assembles - All review

## 6.1 - Master Comparison Table

In [ ]:
# TODO

## 6.2 - All Visualizations

In [ ]:
# TODO

## 6.3 - Conclusions